# Delta Lake Time Travel & Schema Evolution

**Notebook:** `04_delta_time_travel`  
**Lakehouse:** `RetailSalesLakehouse`  
**DP-700 Skill:** Ingest and Transform Data > Batch Data Processing

## Purpose
Demonstrates **Delta Lake** advanced features unique to Microsoft Fabric:

1. **Time travel** — query historical versions of a table
2. **Audit history** — view every change made to a Delta table
3. **RESTORE** — roll back a table to a previous version
4. **Schema evolution** — safely add columns without breaking pipelines
5. **V-ORDER** — Fabric-specific write optimization for faster reads
6. **VACUUM** — reclaim storage from old file versions

These are critical exam topics and real-world Fabric data engineering skills.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

TABLE = "silver.orders"   # Demo table — must exist (run notebook 02 first)
print(f"Delta Lake demo on table: {TABLE}")

## 1. DESCRIBE HISTORY — Full Audit Trail

Every write to a Delta table is recorded in the transaction log (`_delta_log`). 
`DESCRIBE HISTORY` shows every version with who wrote it, when, and what operation was performed.

In [ ]:
history_df = spark.sql(f"DESCRIBE HISTORY {TABLE}")

(
    history_df
    .select("version", "timestamp", "operation", "operationParameters", "numOutputRows")
    .show(20, truncate=False)
)

## 2. Time Travel — Query by Version Number

In [ ]:
# Query the very first version (version 0 = initial write)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table(TABLE)

print(f"Version 0 row count: {df_v0.count():,}")

# Query the latest version for comparison
df_latest = spark.table(TABLE)
print(f"Latest  row count : {df_latest.count():,}")

print(f"\nRows added since v0: {df_latest.count() - df_v0.count():,}")

## 3. Time Travel — Query by Timestamp

Useful for auditing: "What did the data look like at 6 AM this morning before the pipeline ran?"

In [ ]:
from datetime import datetime, timedelta

# Query the table as it was 1 day ago
yesterday = (datetime.utcnow() - timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")

try:
    df_yesterday = (
        spark.read
        .format("delta")
        .option("timestampAsOf", yesterday)
        .table(TABLE)
    )
    print(f"Row count as of {yesterday}: {df_yesterday.count():,}")
except Exception as e:
    # May fail if table is newer than 1 day — that's expected in a fresh setup
    print(f"Note: {e}")
    print("This is expected if the table was created less than 1 day ago.")

## 4. Simulate a Bad Write — Then RESTORE

This demonstrates the recovery workflow: detect a bad load → RESTORE to last known good state.

In [ ]:
# Record the current (good) version before simulating the bad write
good_version = spark.sql(f"""
    SELECT MAX(version) AS v FROM (DESCRIBE HISTORY {TABLE})
""").collect()[0]["v"]

print(f"Current good version: {good_version}")
print(f"Current row count   : {spark.table(TABLE).count():,}")

# Simulate accidental DELETE of a large portion of data
spark.sql(f"DELETE FROM {TABLE} WHERE order_date < '2024-03-01'")
print(f"\nAfter bad DELETE     : {spark.table(TABLE).count():,} rows (BAD STATE)")

# RESTORE to the last good version
spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF {good_version}")
print(f"\nAfter RESTORE v{good_version}    : {spark.table(TABLE).count():,} rows (RESTORED)")

## 5. Schema Evolution — Add a Column Without Breaking Pipelines

Delta Lake supports **schema evolution** — you can add new columns to an existing table
without recreating it or breaking existing readers. Existing rows will show `NULL` for the new column.

In [ ]:
# Check schema before
print("Schema BEFORE evolution:")
spark.table(TABLE).printSchema()

# Add a new column via ALTER TABLE
spark.sql(f"ALTER TABLE {TABLE} ADD COLUMN order_source_system STRING")

print("\nSchema AFTER adding 'order_source_system':")
spark.table(TABLE).printSchema()

# Existing rows have NULL for the new column — no data loss
null_count = spark.table(TABLE).filter(col("order_source_system").isNull()).count()
print(f"\nRows with NULL order_source_system (expected — existing rows): {null_count:,}")

In [ ]:
# Now write new data WITH the new column using mergeSchema
# This is the pattern for incremental loads after a schema change

df_new_data = (
    spark.table(TABLE)
    .limit(100)
    .withColumn("order_source_system", lit("ERP-v2"))   # New column populated
    .withColumn("order_date", lit("2025-01-01").cast("date"))  # New orders
)

(
    df_new_data.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")  # Allow schema evolution on write
    .saveAsTable(TABLE)
)

# Verify: new rows have the value, old rows still have NULL
spark.sql(f"""
    SELECT
        order_source_system,
        COUNT(*) AS row_count
    FROM {TABLE}
    GROUP BY order_source_system
""").show()

# Clean up test column (in production you'd keep it)
spark.sql(f"ALTER TABLE {TABLE} DROP COLUMN order_source_system")
print("Test column dropped — schema restored.")

## 6. V-ORDER — Fabric-Specific Write Optimization

**V-ORDER** is a Microsoft-specific write optimization applied to Parquet files in OneLake.
It enables Power BI DirectLake, SQL analytics endpoint, and Spark to read data **faster**
by pre-sorting and compressing the data during write.

V-ORDER is **enabled by default** in Fabric notebooks. You can verify and control it explicitly.

In [ ]:
# Check current V-ORDER setting
vorder_enabled = spark.conf.get("spark.microsoft.delta.optimizeWrite.enabled", "not set")
vorder_level   = spark.conf.get("spark.microsoft.delta.optimizeWrite.binSize", "not set")

print(f"V-ORDER enabled     : {vorder_enabled}")
print(f"Optimize write bins : {vorder_level}")

# Enable V-ORDER explicitly (it's on by default in Fabric, but shown for awareness)
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.mergeEnabled", "true")

print("\nV-ORDER settings confirmed enabled.")
print("\nWhy V-ORDER matters:")
print("  - Power BI DirectLake reads are significantly faster")
print("  - SQL analytics endpoint query performance improves")
print("  - Applies during any Delta write (append, overwrite, MERGE)")
print("  - Trade-off: slightly slower writes — always worth it for Gold layer")

In [ ]:
# Rewrite the Gold fact_orders table WITH explicit V-ORDER for maximum read performance
# This is the recommended pattern for tables consumed by Power BI

spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")

df_fact = spark.table("gold.fact_orders")

(
    df_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("vorder", "true")   # Explicit V-ORDER hint
    .partitionBy("order_date")
    .saveAsTable("gold.fact_orders")
)

print("gold.fact_orders rewritten with V-ORDER optimization.")
print("This table is now optimized for Power BI DirectLake access.")

## 7. VACUUM — Reclaim Storage from Old Versions

In [ ]:
# VACUUM removes old Delta files that are no longer referenced
# Default retention: 7 days (168 hours) — prevents accidental deletion of in-use files

# DRY RUN first — see what would be deleted without actually deleting
print("DRY RUN — files that would be removed by VACUUM:")
spark.sql(f"VACUUM {TABLE} RETAIN 168 HOURS DRY RUN").show(truncate=False)

# Production VACUUM (168h = 7 days retention)
# spark.sql(f"VACUUM {TABLE} RETAIN 168 HOURS")
# print("VACUUM completed — old Delta files removed.")

print("\nIMPORTANT: VACUUM removes the ability to time-travel beyond the retention period.")
print("Recommended retention: 7 days for production, 1 day for scratch/temp tables.")

## Summary — Delta Lake Capabilities Reference

| Feature | Command | Use Case |
|---|---|---|
| Audit trail | `DESCRIBE HISTORY table` | Compliance, debugging pipeline runs |
| Read old version | `.option("versionAsOf", N)` | Data quality comparison, debugging |
| Read at timestamp | `.option("timestampAsOf", ts)` | Recreate report snapshots |
| Rollback | `RESTORE TABLE t TO VERSION AS OF N` | Recover from bad write/delete |
| Add column | `ALTER TABLE t ADD COLUMN c TYPE` | Schema evolution without re-create |
| Accept new columns | `.option("mergeSchema", "true")` | Incremental writes with new fields |
| V-ORDER | `spark.conf.set("spark.sql.parquet.vorder.enabled", "true")` | Optimize for Power BI DirectLake |
| Compact files | `OPTIMIZE table ZORDER BY (col)` | Reduce file count, improve queries |
| Reclaim storage | `VACUUM table RETAIN 168 HOURS` | Remove files older than retention |
| Preview VACUUM | `VACUUM table DRY RUN` | Check before deleting |